In [ ]:
NETLIST_PATH = "netlists/xerox.json"
NUM_ROWS = 128
NUM_COLS = 128
HIDDEN_DIM = 128
NUM_HIDDEN = 3
NUM_ITERATIONS = 1000
VALUE_LOSS_COEF = 0.5
ENTROPY_COEF = 0.8
CLIP_EPSILON = 0.2
GAMMA = 0.99
LEARNING_RATE = 3e-4

In [ ]:
from src.training import train_VanillaPVN

train_VanillaPVN(
    netlist_path=NETLIST_PATH,
    num_rows=NUM_ROWS,
    num_cols=NUM_COLS,
    hidden_dim=HIDDEN_DIM,
    num_hidden=NUM_HIDDEN,
    gamma=GAMMA,
    clip_epsilon=CLIP_EPSILON,
    value_loss_coef=VALUE_LOSS_COEF,
    entropy_coef=ENTROPY_COEF,
    num_iterations=NUM_ITERATIONS
    )

In [ ]:
from src.training import train_RewardPredictor
from src.netlist import Netlist

netlist = Netlist(NETLIST_PATH)
netlists = {"xerox": netlist}

reward_model, reward_experiment = train_RewardPredictor(
    placements_path="artifacts/2026-08-13_04-14-13/placements.jsonl", 
    netlists=netlists,
    num_rows=NUM_ROWS,
    num_cols=NUM_COLS,
    hidden_channels_e=HIDDEN_DIM,
    num_layers_e=NUM_HIDDEN,
    hidden_channels_r=HIDDEN_DIM,
    num_layers_r=NUM_HIDDEN,
    batch_size=32,
    num_epochs=NUM_ITERATIONS
)

encoder_path = reward_experiment.path / "encoder.pt"
print(encoder_path)

In [ ]:
from src.training import train_GraphPPO

policy_model, ppo_experiment = train_GraphPPO(
    netlist_path=NETLIST_PATH,
    num_rows=NUM_ROWS,
    num_cols=NUM_COLS,
    hidden_channels_e=HIDDEN_DIM,
    num_layers_e=NUM_HIDDEN,
    hidden_dim=HIDDEN_DIM,
    num_hidden=NUM_HIDDEN,
    pretrained_encoder_path=str(encoder_path),
    freeze_encoder=False,
    gamma=GAMMA,
    clip_epsilon=CLIP_EPSILON,
    value_loss_coef=VALUE_LOSS_COEF,
    entropy_coef=ENTROPY_COEF,
    num_iterations=NUM_ITERATIONS
)

In [ ]:
import json
import torch

from src.environment import PlacementEnv
from src.models import GraphPolicyValueNetwork
from src.visualize import compare_placements

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_best_placement(placements_path: str, netlist_name: str):
    with open(placements_path, "r") as f:
        samples = [json.loads(line) for line in f]

    matching = [s for s in samples if s["netlist"] == netlist_name and not s["failed"]]

    if not matching:
        raise ValueError(f"No successful placements found for {netlist_name} in {placements_path}")

    best = min(matching, key=lambda s: s["metrics"]["hpwl"])

    config = {node_id: tuple(coords) for node_id, coords in best["placement"].items()}
    hpwl = best["metrics"]["hpwl"]

    return config, hpwl


def run_deterministic_graph_episode(env: PlacementEnv, model: GraphPolicyValueNetwork, device: torch.device):
    env.reset()
    model.eval()

    done = False
    failed = False

    while not done:
        graph_obs = env.get_graph_observation().to(device)
        action_mask = torch.as_tensor(env.get_action_mask(), dtype=torch.bool, device=device)

        with torch.no_grad():
            logits, _ = model(
                graph_obs.x,
                graph_obs.edge_index,
                graph_obs.edge_weight,
                graph_obs.current_node_idx
            )
            logits = logits.squeeze(0)

            masked_logits = logits.masked_fill(~action_mask, -torch.inf)
            action = torch.argmax(masked_logits).item()

        _, done, failed = env.step(int(action))

    if failed:
        raise RuntimeError("Deterministic rollout failed — no legal placement found.")

    return dict(env.config), env.get_metrics()["hpwl"]

In [ ]:
env = PlacementEnv(netlist=netlist, num_rows=NUM_ROWS, num_cols=NUM_COLS)

in_channels = env.get_graph_observation().x.shape[1]
num_actions = env.num_rows * env.num_cols

policy_model = GraphPolicyValueNetwork(
    output_dim=num_actions,
    in_channels=in_channels,
    hidden_channels_e=HIDDEN_DIM,   
    num_layers_e=NUM_HIDDEN,
    hidden_dim=HIDDEN_DIM,
    num_hidden=NUM_HIDDEN   
).to(device)

graph_ppo_checkpoint_path = "artifacts/2026-08-13_04-27-40/graph_ppo_final.pt"

policy_model.load_state_dict(torch.load(graph_ppo_checkpoint_path, map_location=device))
policy_model.eval()

In [ ]:
config_before, hpwl_before = load_best_placement(
    "artifacts/2026-08-13_04-14-13/placements.jsonl",  # vanilla PVN run
    netlist_name="xerox",
)

config_after, hpwl_after = run_deterministic_graph_episode(env, policy_model, device)

compare_placements(
    netlist=netlist,
    config_before=config_before,
    config_after=config_after,
    num_rows=NUM_ROWS,
    num_cols=NUM_COLS,
    hpwl_before=hpwl_before,
    hpwl_after=hpwl_after,
)

In [ ]:
NEW_NETLIST_PATH = "netlists/ami49.json"
GRAPH_PPO_CHECKPOINT_PATH = "artifacts/2026-08-13_04-27-40/graph_ppo_final.pt"
GRAPH_PPO_CONFIG_PATH = "artifacts/2026-08-13_04-27-40/config.json"

import matplotlib.pyplot as plt
from src.visualize import plot_placement

new_netlist = Netlist(NEW_NETLIST_PATH)
new_env = PlacementEnv(netlist=new_netlist, num_rows=NUM_ROWS, num_cols=NUM_COLS)

with open(GRAPH_PPO_CONFIG_PATH, "r") as f:
    train_config = json.load(f)

in_channels = new_env.get_graph_observation().x.shape[1]
num_actions = new_env.num_rows * new_env.num_cols

policy_model = GraphPolicyValueNetwork(
    output_dim=num_actions,
    in_channels=in_channels,
    hidden_channels_e=train_config.get("hidden_channels_e", 128),
    num_layers_e=train_config.get("num_layers_e", 3),
    hidden_dim=train_config.get("hidden_dim", 128),
    num_hidden=train_config.get("num_hidden", 3),
).to(device)

policy_model.load_state_dict(torch.load(GRAPH_PPO_CHECKPOINT_PATH, map_location=device))
policy_model.eval()

# Deterministic rollout on the new netlist
new_env.reset()
done, failed = False, False

while not done:
    graph_obs = new_env.get_graph_observation().to(device)
    action_mask = torch.as_tensor(new_env.get_action_mask(), dtype=torch.bool, device=device)

    with torch.no_grad():
        logits, _ = policy_model(
            graph_obs.x, graph_obs.edge_index, graph_obs.edge_weight, graph_obs.current_node_idx,
        )
        masked_logits = logits.squeeze(0).masked_fill(~action_mask, -torch.inf)
        action = torch.argmax(masked_logits).item()

    _, done, failed = new_env.step(int(action))

if failed:
    raise RuntimeError("No legal placement found on this netlist.")

hpwl_new = new_env.get_metrics()["hpwl"]

fig, ax = plt.subplots(figsize=(7, 7))
plot_placement(new_netlist, new_env.config, NUM_ROWS, NUM_COLS, title=f"Trained Policy on New Netlist (HPWL: {hpwl_new:.2f})", ax=ax)
plt.tight_layout()
plt.show()